# Document Question Answering System (RAG)

A Retrieval-Augmented Generation pipeline that answers questions grounded in your own documents (PDFs, text files, or a Hugging Face dataset) instead of relying only on a language model's internal knowledge.

**Pipeline stages:**
1. Document Ingestion
2. Text Chunking
3. Embedding Creation
4. Vector Database Storage (Chroma)
5. Query Embedding
6. Context Retrieval
7. Answer Generation (local LLM via Ollama)


## 0. Setup

Install dependencies (skip if already installed).

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys
sys.path.append("..")  # so we can import from src/

from src.data_loader import load_folder, load_huggingface_dataset
from src.chunker import chunk_documents
from src.vector_store import get_embedding_model, build_vector_store, load_vector_store, retrieve_relevant_chunks
from src.rag import answer_question, generate_answer, build_prompt

## 1. Document Ingestion

Choose **one** of the two options below depending on your dataset.

- **Option A** — Load your own PDFs / text files from `data/raw/`
- **Option B** — Load a Hugging Face QA dataset (e.g. `squad`, `vectara/open_ragbench`)


In [ ]:
# --- Option A: your own documents ---
documents = load_folder("../data/raw")
print(f"Loaded {len(documents)} document(s)")
if documents:
    print(documents[0]["source"], "->", documents[0]["text"][:200], "...")

In [ ]:
# --- Option B: Hugging Face dataset (uncomment to use instead) ---
# documents = load_huggingface_dataset(dataset_name="squad", split="train", max_docs=200)
# print(f"Loaded {len(documents)} document(s)")
# print(documents[0]["source"], "->", documents[0]["text"][:200], "...")

## 2. Text Chunking

Splitting text into smaller overlapping chunks improves retrieval precision — a whole document is too large and noisy to embed as a single vector.


In [ ]:
chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=50)
print(f"Produced {len(chunks)} chunks from {len(documents)} document(s)")
print(chunks[0])

## 3. Embedding Creation + 4. Vector Database

Each chunk is converted into a 384-dimensional vector using `sentence-transformers/all-MiniLM-L6-v2`, then stored in a persistent Chroma collection for similarity search.


In [ ]:
embedding_model = get_embedding_model()

vector_store = build_vector_store(
    chunks,
    persist_directory="../chroma_db",
    collection_name="rag_docs",
    embedding_model=embedding_model,
)
print("Vector store built and persisted to ../chroma_db")

> Already built your index before? Skip step 3 and reload it instead:
> ```python
> vector_store = load_vector_store(persist_directory="../chroma_db")
> ```


## 5. Query Processing + 6. Context Retrieval

The user's question is embedded with the same model, then compared against stored chunk vectors to find the most relevant context.


In [ ]:
query = "What is the main idea of the document?"

retrieved_chunks = retrieve_relevant_chunks(vector_store, query, k=4)

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"--- Retrieved chunk {i} (source: {chunk.metadata.get('source')}) ---")
    print(chunk.page_content[:300], "...\n")

## 7. Answer Generation

The retrieved chunks are inserted into a prompt as grounding context, then a local LLM (via [Ollama](https://ollama.com)) generates the final answer.

Before running this cell, make sure Ollama is installed and running, and pull a model once:
```bash
ollama pull llama3
```


In [ ]:
answer = generate_answer(query, retrieved_chunks, model_name="llama3")
print("Question:", query)
print("\nAnswer:", answer)

## End-to-End: Ask Any Question

Once the vector store is built, use this single helper for any new question.

In [ ]:
def ask(question, k=4, model_name="llama3"):
    result = answer_question(vector_store, question, k=k, model_name=model_name)
    print("Question:", result["question"])
    print("\nAnswer:", result["answer"])
    print("\nSources used:")
    for s in result["sources"]:
        print(" -", s)
    return result

_ = ask("Summarize the key points of the document.")